In [10]:
# ── STEP: LIME Explanation Layer ──
# (Run this once in your terminal first: pip install lime)

from lime.lime_text import LimeTextExplainer
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
# ... rest of your code stays the same

In [3]:
import pandas as pd

# 1. Load the files
df_spam_assassin = pd.read_csv(r'D:\Cyber Defense Hub website\data\completeSpamAssassin.csv')
df_enron = pd.read_csv(r'D:\Cyber Defense Hub website\data\emails.csv')
df_phishing = pd.read_csv(r'D:\Cyber Defense Hub website\data\phishing_email.csv')

# 2. Standardize Enron (Based on your screenshot: ['file', 'message'])
# We label these as 0 (Ham) because this specific csv is usually the 'Ham' portion
df_enron = df_enron[['message']].rename(columns={'message': 'text'})
df_enron['label'] = 0 

# 3. Standardize SpamAssassin (Based on your screenshot: ['Body', 'Label'])
df_spam_assassin = df_spam_assassin[['Body', 'Label']].rename(columns={'Body': 'text', 'Label': 'label'})

# 4. Standardize Phishing (Based on your screenshot: ['text_combined', 'label'])
df_phishing = df_phishing[['text_combined', 'label']].rename(columns={'text_combined': 'text'})

# 5. Combine and Shuffle
master_df = pd.concat([df_enron, df_spam_assassin, df_phishing], ignore_index=True)

# Clean up: remove empty rows and shuffle for training
master_df = master_df.dropna(subset=['text']).sample(frac=1).reset_index(drop=True)

print("✅ Master Dataset Ready!")
print(f"Total rows: {len(master_df)}")
print(master_df['label'].value_counts())

✅ Master Dataset Ready!
Total rows: 605932
label
0    561146
1     44786
Name: count, dtype: int64


In [4]:
# Check the actual names of the columns in your files
print("Enron Columns:", df_enron.columns.tolist())
print("SpamAssassin Columns:", df_spam_assassin.columns.tolist())
print("Phishing Columns:", df_phishing.columns.tolist())

Enron Columns: ['text', 'label']
SpamAssassin Columns: ['text', 'label']
Phishing Columns: ['text', 'label']


In [3]:
from collections import Counter
import re

def get_common_words(label, num=10):
    text = " ".join(master_df[master_df['label'] == label]['text'].astype(str)).lower()
    words = re.findall(r'\w+', text)
    # Filter out common 'stop words' manually for now
    stop_words = {'the', 'to', 'and', 'a', 'in', 'is', 'it', 'you', 'of', 'for', 'on', 'this'}
    meaningful_words = [w for w in words if w not in stop_words and len(w) > 3]
    return Counter(meaningful_words).most_common(num)

print("🚩 Top Phishing Words:", get_common_words(1))
print("✅ Top Safe Words:", get_common_words(0))

🚩 Top Phishing Words: [('2008', 32009), ('email', 28437), ('please', 19449), ('money', 19279), ('company', 18761), ('account', 16653), ('news', 16149), ('time', 15259), ('cnncom', 14533), ('information', 14021)]
✅ Top Safe Words: [('enron', 7492915), ('from', 1802599), ('recipients', 1131342), ('that', 1107853), ('content', 1062337), ('subject', 983669), ('2001', 839050), ('message', 782912), ('with', 776084), ('will', 750968)]


In [5]:
import re

def rule_engine_analysis(text):
    reasons = []
    score = 0
    text_lower = str(text).lower()
    
    # 🚩 Rule 1: Urgency & Pressure
    # FIXED: Changed variable name to match the loop below
    urgency_keywords = ['verify', 'suspended', 'immediately', 'action required', 'unauthorized', 'urgent']
    found_urgency = [word for word in urgency_keywords if word in text_lower]
    if found_urgency:
        reasons.append(f"Urgency triggers found: {', '.join(found_urgency)}")
        score += 20
        
    # 🚩 Rule 2: Financial/Money Lures
    money_words = ['bank', 'transfer', 'invoice', 'payment', 'beneficiary', 'wire']
    found_money = [word for word in money_words if word in text_lower]
    if found_money:
        reasons.append(f"Financial keywords detected: {', '.join(found_money)}")
        score += 15

    # 🚩 Rule 3: Suspicious Links (Simple Pattern)
    if "http" in text_lower or "www" in text_lower:
        if "click here" in text_lower or "login" in text_lower:
            reasons.append("Contains suspicious call-to-action link")
            score += 25
            
    return score, reasons

# Test it again
sample_phish = master_df[master_df['label'] == 1]['text'].iloc[0]
score, reasons = rule_engine_analysis(sample_phish)
print(f"Risk Score: {score}\nReasons: {reasons}")

Risk Score: 0
Reasons: []


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Use a 50k sample if the 1.3GB is making your laptop laggy
train_df = master_df.sample(50000, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(train_df['text'], train_df['label'], test_size=0.2)

# Convert text to numbers (TF-IDF)
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train simple Logistic Regression
baseline_model = LogisticRegression()
baseline_model.fit(X_train_tfidf, y_train)

# Check Accuracy
predictions = baseline_model.predict(X_test_tfidf)
print(f"Baseline Accuracy: {accuracy_score(y_test, predictions)*100:.2f}%")
print(classification_report(y_test, predictions))

Baseline Accuracy: 98.94%
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      9217
           1       0.97      0.89      0.93       783

    accuracy                           0.99     10000
   macro avg       0.98      0.95      0.96     10000
weighted avg       0.99      0.99      0.99     10000



In [7]:
import Levenshtein # You might need to pip install python-Levenshtein

def rule_engine(email_text, sender_email=""):
    reasons = []
    score = 0
    text = str(email_text).lower()
    
    # 🚩 Rule 1: Urgency & Pressure
    urgency_keywords = ['immediately', 'action required', 'suspended', 'verify', 'unauthorized']
    found_urgency = [w for w in urgency_keywords if w in text]
    if found_urgency:
        reasons.append(f"Urgency detected: {', '.join(found_urgency)}")
        score += 30

    # 🚩 Rule 2: Financial Lures
    financial_keywords = ['wire transfer', 'payment', 'bank account', 'invoice', 'inheritance']
    found_finance = [w for w in financial_keywords if w in text]
    if found_finance:
        reasons.append(f"Financial lure: {', '.join(found_finance)}")
        score += 25

    # 🚩 Rule 3: Lookalike Domains (The "Levenshtein" Rule)
    # If a domain is 1 character off from a famous brand, it's a phishing trick
    trusted_brands = ['paypal.com', 'microsoft.com', 'google.com', 'amazon.com']
    if "@" in sender_email:
        sender_domain = sender_email.split('@')[-1]
        for brand in trusted_brands:
            distance = Levenshtein.distance(sender_domain, brand)
            if 0 < distance <= 2: # e.g., 'paypa1.com' is 1 edit from 'paypal.com'
                reasons.append(f"Lookalike domain: '{sender_domain}' mimics '{brand}'")
                score += 40
    
    return score, reasons

# TEST IT:
test_score, test_reasons = rule_engine("Please verify your bank account immediately!", "support@paypa1.com")
print(f"Risk Score: {test_score}/100")
print(f"Reasons: {test_reasons}")

Risk Score: 95/100
Reasons: ['Urgency detected: immediately, verify', 'Financial lure: bank account', "Lookalike domain: 'paypa1.com' mimics 'paypal.com'"]


In [7]:
from transformers import AutoTokenizer
from datasets import Dataset

# Load the DistilBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Create a balanced 20k sample for training
bert_sample = master_df.sample(20000, random_state=42)
train_ds = Dataset.from_pandas(bert_sample[['text', 'label']])

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

# Tokenize the whole dataset
tokenized_ds = train_ds.map(preprocess_function, batched=True)
tokenized_ds = tokenized_ds.train_test_split(test_size=0.2)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [9]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Load the model for binary classification
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# 2. Optimized arguments for your 16GB RAM and 2GB GPU
training_args = TrainingArguments(
    output_dir="./phishscan_results",
    eval_strategy="epoch",        # Use the new naming convention
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8, 
    per_device_eval_batch_size=8,
    num_train_epochs=3,            
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none"              # Prevents errors with missing tracking tools
)

# 3. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
)

# 4. START THE TRAINING
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.034804,0.032645
2,0.009856,0.032920
3,0.000691,0.044471


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\yashv\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\yashv\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=6000, training_loss=0.020115784004330634, metrics={'train_runtime': 70020.5048, 'train_samples_per_second': 0.686, 'train_steps_per_second': 0.086, 'total_flos': 6358435135488000.0, 'train_loss': 0.020115784004330634, 'epoch': 3.0})

In [8]:
# ── STEP: LIME Explanation Layer ──────────────────────────
!pip install lime  # run once in terminal: pip install lime

from lime.lime_text import LimeTextExplainer
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np

# Load your saved model
tokenizer = AutoTokenizer.from_pretrained("./phishscan_final_model")
model = AutoModelForSequenceClassification.from_pretrained("./phishscan_final_model")
model.eval()

# Prediction function LIME needs
def predict_proba(texts):
    inputs = tokenizer(
        texts, return_tensors="pt",
        truncation=True, padding=True, max_length=256
    )
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1).numpy()
    return probs  # shape: (n_texts, 2)

# Build explainer
explainer = LimeTextExplainer(class_names=["Safe", "Phishing"])

# Test on a sample phishing email
sample_text = master_df[master_df['label'] == 1]['text'].iloc[0]

explanation = explainer.explain_instance(
    sample_text[:500],     # LIME works on shorter text
    predict_proba,
    num_features=10,       # top 10 words that influenced the decision
    num_samples=100        # how many perturbations to try
)

# Print word-level explanation
print("=== LIME EXPLANATION ===")
for word, weight in explanation.as_list():
    direction = "🚩 PHISHING" if weight > 0 else "✅ SAFE"
    print(f"  {direction}  '{word}'  (weight: {weight:.4f})")

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

=== LIME EXPLANATION ===
  🚩 PHISHING  'currency'  (weight: 0.0044)
  ✅ SAFE  '2002'  (weight: -0.0044)
  ✅ SAFE  'using'  (weight: -0.0035)
  ✅ SAFE  'linnea2278102emailis'  (weight: -0.0033)
  🚩 PHISHING  'importan'  (weight: 0.0028)
  ✅ SAFE  'means'  (weight: -0.0022)
  🚩 PHISHING  'free'  (weight: 0.0021)
  🚩 PHISHING  '1st'  (weight: 0.0017)
  ✅ SAFE  'use'  (weight: -0.0016)
  ✅ SAFE  'market'  (weight: -0.0012)


In [13]:
# ── FULL PREDICTION FUNCTION ──────────────────────────────
import Levenshtein

URGENCY_WORDS   = ['immediately','action required','suspended','verify',
                   'unauthorized','urgent','account locked','expire']
FINANCIAL_WORDS = ['wire transfer','payment','bank account',
                   'invoice','inheritance','beneficiary']
TRUSTED_BRANDS  = ['paypal.com','microsoft.com','google.com',
                   'amazon.com','apple.com','netflix.com']

def rule_engine(text, sender_email=""):
    reasons, score = [], 0
    t = text.lower()

    # Rule 1: Urgency
    found_u = [w for w in URGENCY_WORDS if w in t]
    if found_u:
        reasons.append(f"Urgency triggers: {', '.join(found_u)}")
        score += 30

    # Rule 2: Financial lures
    found_f = [w for w in FINANCIAL_WORDS if w in t]
    if found_f:
        reasons.append(f"Financial lure detected: {', '.join(found_f)}")
        score += 25

    # Rule 3: Lookalike domain
    if sender_email and "@" in sender_email:
        domain = sender_email.split('@')[-1].lower()
        for brand in TRUSTED_BRANDS:
            dist = Levenshtein.distance(domain, brand)
            if 0 < dist <= 2:
                reasons.append(f"Lookalike domain '{domain}' mimics '{brand}'")
                score += 40

    # Rule 4: Suspicious link pattern
    if ("http" in t or "www" in t) and ("click here" in t or "login" in t):
        reasons.append("Suspicious call-to-action link detected")
        score += 20

    return min(score, 100), reasons


def phishscan_predict(email_text, sender_email=""):
    """
    Full 4-layer PhishScan pipeline:
    Layer 1 → Rule Engine   (fast flags)
    Layer 2 → DistilBERT    (ML probability)
    Layer 3 → LIME          (word explanations)
    Layer 4 → Ensemble      (final 0-100 score)
    """
    # ── Layer 1: Rules ──
    rule_score, rule_reasons = rule_engine(email_text, sender_email)

    # ── Layer 2: BERT ──
    inputs = tokenizer(
        [email_text[:512]], return_tensors="pt",
        truncation=True, padding=True, max_length=256
    )
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1).numpy()[0]
    bert_phish_prob = float(probs[1])

    # ── Layer 3: LIME explanations ──
    exp = explainer.explain_instance(
        email_text[:400], predict_proba,
        num_features=5, num_samples=50
    )
    lime_reasons = [
        f"Suspicious word: '{w}' (weight: {wt:.3f})"
        for w, wt in exp.as_list() if wt > 0.005
    ]

    # ── Layer 4: Ensemble Score ──
    bert_contribution = int(bert_phish_prob * 60)   # BERT  → max 60 pts
    rule_contribution  = min(rule_score, 40)         # Rules → max 40 pts
    final_score = min(bert_contribution + rule_contribution, 100)

    # Verdict
    if final_score >= 70:
        verdict = "HIGH RISK — LIKELY PHISHING"
    elif final_score >= 40:
        verdict = "MEDIUM RISK — SUSPICIOUS"
    else:
        verdict = "LOW RISK — LIKELY SAFE"

    return {
        "score":      final_score,
        "verdict":    verdict,
        "confidence": round(bert_phish_prob * 100, 1),
        "reasons":    (rule_reasons + lime_reasons) or ["No specific indicators found"]
    }


# ── TEST IT ──
result = phishscan_predict(
    "Your account has been suspended. Verify immediately to avoid permanent closure.",
    "support@paypa1.com"
)

print("=" * 40)
print(f"  RISK SCORE : {result['score']} / 100")
print(f"  VERDICT    : {result['verdict']}")
print(f"  CONFIDENCE : {result['confidence']}%")
print("=" * 40)
print("  REASONS:")
for r in result['reasons']:
    print(f"    >> {r}")

  RISK SCORE : 69 / 100
  VERDICT    : MEDIUM RISK — SUSPICIOUS
  CONFIDENCE : 48.6%
  REASONS:
    >> Urgency triggers: immediately, suspended, verify
    >> Lookalike domain 'paypa1.com' mimics 'paypal.com'
    >> Suspicious word: 'suspended' (weight: 0.006)


In [10]:
# Save the trained model and tokenizer
trainer.save_model("./phishscan_final_model")
tokenizer.save_pretrained("./phishscan_final_model")
print("✅ Success! Your AI is saved in the 'phishscan_final_model' folder.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Success! Your AI is saved in the 'phishscan_final_model' folder.


In [11]:
import os
print(os.getcwd())

C:\Users\yashv
